# HGTCM P4 - Score Raw OCR and BART Corrections

This notebook scores the canonical target-only OCR result, the Tesseract Gaelic+Latin baseline, and the frozen/fine-tuned BART correction controls. The archived An Gaodhal donor branch is intentionally excluded from the canonical target-study scorecards unless explicitly re-enabled.

## Imports


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from pathlib import Path
import random

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 110)


def summary_table(summary, title=None):
    if title:
        display(Markdown(f'### {title}'))
    display(pd.DataFrame([summary]))



## Visible Code: Inference and Evaluation Utilities

Inference, CER/WER, and report-writing logic live directly in this notebook so the analysis remains auditable.



In [ ]:
# ## Workspace, scoring metrics, and large analysis + optional TrOCR infer over checkpoints.
# Why: scoring consolidates model, split, and cleanliness factors so comparisons are apples-to-apples.
import os
import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

from PIL import Image

@dataclass(frozen=True)
class Workspace:
    root: Path

    @property
    def angaodhal_page_zip(self) -> Path:
        return self.root / "AnGaodhal_transkribusPage.zip"

    @property
    def angaodhal_alto_zip(self) -> Path:
        return self.root / "AnGaodhal_alto.zip"

    @property
    def angaodhal_metadata_csv(self) -> Path:
        return self.root / "AnGaodhal_pageMetadata.csv"

    @property
    def target_gt_zip(self) -> Path:
        matches = sorted(self.root.glob("GroundTruth811-*.zip"))
        return matches[0] if matches else self.root / "GroundTruth811-20260425T073039Z-3-001.zip"

    @property
    def target_image_zips(self) -> list[Path]:
        return sorted(self.root.glob("drive-download-*.zip"))

    @property
    def data_dir(self) -> Path:
        return self.root / "data"

    @property
    def manifests_dir(self) -> Path:
        return self.root / "manifests"

    @property
    def runs_dir(self) -> Path:
        return self.root / "runs"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"


def workspace_from(value: str | None = None) -> Workspace:
    return Workspace(Path(value or ".").resolve())


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path: Path, payload: object) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return path


def write_jsonl(path: Path, rows: list[dict]) -> Path:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def normalize_text(text: str) -> str:
    return " ".join((text or "").replace("\r\n", "\n").replace("\r", "\n").split())


def env_int_or_none(name: str, default: int | None = None) -> int | None:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return default
    if raw.strip().lower() in {"none", "all", "full"}:
        return None
    return int(raw)


def env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


INCLUDE_ARCHIVE_REFERENCE_RESULTS = env_flag("INCLUDE_ARCHIVE_REFERENCE_RESULTS", "0")


WIN_MANIFEST_PREFIX = "C:\\Users\\Jobias\\OneDrive\\Desktop\\Project Decode\\"
PROJECT_ROOT = Path(os.environ.get("PROJECT_DECODE_ROOT", Path.cwd())).resolve()


def portable_path(path_value: str | Path) -> Path:
    """Rewrite Windows-absolute manifest paths under PROJECT_DECODE_ROOT."""

    raw = str(path_value)
    if raw.startswith(WIN_MANIFEST_PREFIX):
        rel_parts = raw[len(WIN_MANIFEST_PREFIX):].split("\\")
        return PROJECT_ROOT.joinpath(*rel_parts)
    path = Path(raw)
    if path.exists():
        return path
    if not path.is_absolute():
        return PROJECT_ROOT / path
    parts = path.parts
    if "data" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("data"):])
    if "manifests" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("manifests"):])
    if "runs" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("runs"):])
    if "reports" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("reports"):])
    return path


def resolve_image_path(row: dict, workspace: Workspace | None = None) -> Path:
    image_path = portable_path(row["image_path"])
    if image_path.exists():
        return image_path
    raise FileNotFoundError(f"Could not resolve image path for {row.get('sample_id')}: {row.get('image_path')} -> {image_path}")

def levenshtein(left: list | str, right: list | str) -> int:
    if left == right:
        return 0
    if not left:
        return len(right)
    if not right:
        return len(left)
    previous = list(range(len(right) + 1))
    for left_index, left_item in enumerate(left, start=1):
        current = [left_index]
        for right_index, right_item in enumerate(right, start=1):
            current.append(min(current[right_index - 1] + 1, previous[right_index] + 1, previous[right_index - 1] + (left_item != right_item)))
        previous = current
    return previous[-1]


def cer(reference: str, prediction: str) -> float:
    return levenshtein(reference or "", prediction or "") / max(1, len(reference or ""))


def wer(reference: str, prediction: str) -> float:
    return levenshtein((reference or "").split(), (prediction or "").split()) / max(1, len((reference or "").split()))


def infer_trocr(
    *,
    manifest_path: Path,
    model_dir: Path,
    output_path: Path,
    split: str | None = "test",
    batch_size: int = 2,
    max_target_length: int = 256,
    max_samples: int | None = None,
    device: str = "auto",
) -> dict:
    torch, DataLoader, Dataset, TrOCRProcessor, VisionEncoderDecoderModel = _require_training_deps()
    rows = _select_rows(read_jsonl(manifest_path), split, max_samples)
    processor = TrOCRProcessor.from_pretrained(model_dir)
    model = VisionEncoderDecoderModel.from_pretrained(model_dir)
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id
    model.generation_config.max_length = max_target_length
    selected_device = "cuda" if device == "auto" and torch.cuda.is_available() else "cpu" if device == "auto" else device
    if selected_device.startswith("cuda"):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    model.to(selected_device)
    model.eval()

    class ImageDataset(Dataset):
        def __len__(self) -> int:
            return len(rows)

        def __getitem__(self, idx: int) -> dict:
            row = rows[idx]
            image = Image.open(resolve_image_path(row)).convert("RGB")
            return {"pixel_values": processor(image, return_tensors="pt").pixel_values.squeeze(0), "row": row}

    def collate(batch: list[dict]) -> dict:
        return {"pixel_values": torch.stack([item["pixel_values"] for item in batch]), "rows": [item["row"] for item in batch]}

    predictions: list[dict] = []
    loader = DataLoader(ImageDataset(), batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=2 if selected_device.startswith("cuda") else 0, pin_memory=selected_device.startswith("cuda"))
    with torch.inference_mode():
        for batch in loader:
            generated_ids = model.generate(batch["pixel_values"].to(selected_device, non_blocking=True), max_length=max_target_length)
            texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for row, prediction in zip(batch["rows"], texts):
                predictions.append(
                    {
                        "sample_id": row["sample_id"],
                        "page_id": row["page_id"],
                        "group_id": row["group_id"],
                        "dataset_split": row.get("dataset_split"),
                        "model_dir": str(model_dir),
                        "prediction_raw": prediction,
                        "prediction_normalized": normalize_text(prediction),
                        "reference_normalized": row.get("normalized_text", ""),
                        "status": "ok",
                    }
                )
    write_jsonl(output_path, predictions)
    summary = {"manifest_path": str(manifest_path), "model_dir": str(model_dir), "output_path": str(output_path), "split": split, "sample_count": len(predictions), "device": selected_device}
    write_json(output_path.with_suffix(".summary.json"), summary)
    return summary


def evaluate_predictions(
    *,
    manifest_path: Path,
    predictions_path: Path,
    output_dir: Path,
    page_clean_only: bool = False,
    suffix: str = "",
) -> dict:
    manifest_rows = {row["sample_id"]: row for row in read_jsonl(manifest_path)}
    per_sample: list[dict] = []
    for pred in read_jsonl(predictions_path):
        sample_id = pred.get("sample_id", "")
        manifest_row = manifest_rows.get(sample_id, {})
        if page_clean_only and not manifest_row.get("page_clean"):
            continue
        reference = pred.get("reference_normalized") or manifest_row.get("normalized_text", "")
        prediction = pred.get("prediction_normalized") or pred.get("prediction_raw") or ""
        per_sample.append(
            {
                "sample_id": sample_id,
                "page_id": pred.get("page_id") or manifest_row.get("page_id", ""),
                "dataset_split": pred.get("dataset_split") or manifest_row.get("dataset_split", ""),
                "page_clean": bool(manifest_row.get("page_clean")),
                "reference": reference,
                "prediction": prediction,
                "cer": cer(reference, prediction),
                "wer": wer(reference, prediction),
            }
        )
    by_page: dict[str, list[dict]] = defaultdict(list)
    for row in per_sample:
        by_page[row["page_id"]].append(row)
    per_page = [
        {"page_id": page_id, "sample_count": len(rows), "mean_cer": sum(row["cer"] for row in rows) / len(rows), "mean_wer": sum(row["wer"] for row in rows) / len(rows)}
        for page_id, rows in sorted(by_page.items())
    ]
    overall = {
        "manifest_path": str(manifest_path),
        "predictions_path": str(predictions_path),
        "evaluation_subset": "page_clean" if page_clean_only else "line_clean",
        "sample_count": len(per_sample),
        "page_count": len(per_page),
        "mean_cer": sum(row["cer"] for row in per_sample) / max(1, len(per_sample)),
        "mean_wer": sum(row["wer"] for row in per_sample) / max(1, len(per_sample)),
    }
    ensure_dir(output_dir)
    write_jsonl(output_dir / f"per_sample_scores{suffix}.jsonl", per_sample)
    write_jsonl(output_dir / f"per_page_scores{suffix}.jsonl", per_page)
    write_json(output_dir / f"overall_summary{suffix}.json", overall)
    return overall


def _require_training_deps():
    try:
        import torch
        from torch.utils.data import DataLoader, Dataset
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    except ImportError as exc:
        raise SystemExit("Install training dependencies with: python -m pip install -r requirements-trocr.txt") from exc
    return torch, DataLoader, Dataset, TrOCRProcessor, VisionEncoderDecoderModel


def _is_trainable_crop(row: dict) -> bool:
    return row.get("line_trainable", row.get("crop_quality", {}).get("status", "ok") == "ok")


def _select_rows(rows: list[dict], split: str | None, limit: int | None) -> list[dict]:
    selected = [row for row in rows if (not split or row.get("dataset_split") == split) and _is_trainable_crop(row)]
    return selected[:limit] if limit else selected




## Workspace Configuration


In [ ]:
WORKSPACE = workspace_from(os.environ.get('PROJECT_DECODE_ROOT', '.'))
DATA_RUN_LABEL = os.environ.get('DATA_RUN_LABEL', 'pilot').strip() or 'pilot'
RUN_INFERENCE = os.environ.get('RUN_SCORE_INFERENCE', '1').strip().lower() in {'1', 'true', 'yes', 'on'}

summary_table({
    'Workspace': str(WORKSPACE.root),
    'Data run label': DATA_RUN_LABEL,
    'Run raw OCR inference scoring': RUN_INFERENCE,
}, 'Scoring configuration')


## Load Test Data


In [ ]:
target_manifest = WORKSPACE.manifests_dir / f'target_lines_{DATA_RUN_LABEL}.jsonl'
target_rows_all = read_jsonl(target_manifest)
target_rows = _select_rows(target_rows_all, None, None)
target_df = pd.DataFrame(target_rows)
if target_df.empty:
    print('No accepted target rows yet. Return to P1 and inspect target preparation.')
else:
    display(Markdown(f'Target manifest: `{target_manifest}`'))
    display(target_df['dataset_split'].value_counts())
    display(pd.DataFrame([{
        'all_rows': len(target_rows_all),
        'line_clean_rows': len(target_rows),
        'page_clean_rows': sum(1 for row in target_rows if row.get('page_clean')),
        'line_clean_pages': target_df['page_id'].nunique(),
        'page_clean_pages': target_df[target_df.get('page_clean', False) == True]['page_id'].nunique() if 'page_clean' in target_df else 0,
    }]))
    display(target_df.head())


## Load Best Performing Model


In [ ]:
# ## Normalize run folder names; pick one checkpoint per config for fair comparison.
# Why: grouping timestamped runs by config avoids selection bias from duplicate retries.
import re


def run_config_name(run_name: str) -> str:
    """Collapse timestamped/recovery run names to their experiment role."""
    name = re.sub(r"_\d{8}T\d{6}Z$", "", run_name)
    name = name.replace("_recovery", "")
    return name


def latest_checkpoint_per_config(paths: list[Path]) -> list[Path]:
    latest: dict[str, Path] = {}
    for checkpoint_path in sorted(paths):
        run_name = checkpoint_path.parent.name
        config = run_config_name(run_name)
        if config not in latest or run_name > latest[config].parent.name:
            latest[config] = checkpoint_path
    return [latest[key] for key in sorted(latest)]


checkpoint_roots = [WORKSPACE.runs_dir]
for raw_root in os.environ.get('EXTRA_CHECKPOINT_ROOTS', '').split(os.pathsep):
    raw_root = raw_root.strip()
    if raw_root:
        checkpoint_roots.append(Path(raw_root))

candidate_checkpoints_all = []
for checkpoint_root in checkpoint_roots:
    candidate_checkpoints_all.extend(sorted(checkpoint_root.glob(f'trocr_*_{DATA_RUN_LABEL}_*/*best_checkpoint')))
candidate_checkpoints_all = sorted(set(candidate_checkpoints_all))
candidate_checkpoints = latest_checkpoint_per_config(candidate_checkpoints_all)
print('All candidate checkpoints:', len(candidate_checkpoints_all))
print('Latest checkpoint per config:')
for path in candidate_checkpoints:
    print(run_config_name(path.parent.name), '->', path)


## Evaluate on Test Set


In [ ]:
# ## Optional: run TrOCR inference for each candidate checkpoint on test split.
# Why: evaluating identical test rows across checkpoints yields fair generalization ranking.
eval_summaries = []

if RUN_INFERENCE and candidate_checkpoints and target_rows:
    for model_dir in candidate_checkpoints:
        run_name = Path(model_dir).parent.name
        predictions_path = Path(model_dir).parent / 'predictions_test.jsonl'
        infer_summary = infer_trocr(
            manifest_path=target_manifest,
            model_dir=Path(model_dir),
            output_path=predictions_path,
            split='test',
            batch_size=env_int_or_none('OCR_EVAL_BATCH_SIZE', 64 if os.environ.get('TRAINING_PROFILE', '').strip().lower() in {'h200', 'full'} else 32),
            max_samples=None,
            device='auto',
        )
        eval_summary = evaluate_predictions(
            manifest_path=target_manifest,
            predictions_path=predictions_path,
            output_dir=WORKSPACE.reports_dir / run_name,
        )
        page_clean_summary = evaluate_predictions(
            manifest_path=target_manifest,
            predictions_path=predictions_path,
            output_dir=WORKSPACE.reports_dir / run_name,
            page_clean_only=True,
            suffix='_page_clean',
        )
        eval_summary['run_name'] = run_name
        eval_summary['config_name'] = run_config_name(run_name)
        eval_summary['model_dir'] = str(model_dir)
        eval_summary['predictions_path'] = str(predictions_path)
        eval_summary['page_clean_sample_count'] = page_clean_summary['sample_count']
        eval_summary['page_clean_page_count'] = page_clean_summary['page_count']
        eval_summary['page_clean_mean_cer'] = page_clean_summary['mean_cer']
        eval_summary['page_clean_mean_wer'] = page_clean_summary['mean_wer']
        eval_summaries.append(eval_summary)

    comparison_df = pd.DataFrame(eval_summaries)[[
        'config_name', 'run_name', 'sample_count', 'page_count', 'mean_cer', 'mean_wer',
        'page_clean_sample_count', 'page_clean_page_count', 'page_clean_mean_cer', 'page_clean_mean_wer',
        'predictions_path',
    ]]
    comparison_path = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_final_raw_ocr_comparison.csv'
    ensure_dir(WORKSPACE.reports_dir)
    comparison_df.to_csv(comparison_path, index=False)
    display(comparison_df)
    display(Markdown(f'Comparison CSV written to `{comparison_path}`.'))
elif not target_rows:
    display(Markdown('Inference skipped because there are no target rows.'))
elif not candidate_checkpoints:
    display(Markdown('Inference skipped because no checkpoints were found. Run Part 2 first.'))
else:
    display(Markdown('Inference skipped. Set `RUN_INFERENCE=True` to evaluate checkpoints.'))


## Load Existing Evaluation Reports


In [ ]:
# ## Collect overall_summary.json from prior runs, merge page_clean stats when present.
def include_raw_ocr_summary(run_name: str) -> bool:
    if f'_{DATA_RUN_LABEL}_' not in run_name or run_name.startswith('correction_'):
        return False
    if run_name.startswith('trocr_target_only_'):
        return True
    if INCLUDE_ARCHIVE_REFERENCE_RESULTS and (
        run_name.startswith('trocr_target_from_reference_') or run_name.startswith('trocr_angaodhal_')
    ):
        return True
    return False


summary_paths = sorted(WORKSPACE.reports_dir.glob('*/overall_summary.json'))
summaries = []
for path in summary_paths:
    run_name = path.parent.name
    if not include_raw_ocr_summary(run_name):
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    page_clean_path = path.parent / 'overall_summary_page_clean.json'
    if page_clean_path.exists():
        page_clean = json.loads(page_clean_path.read_text(encoding='utf-8'))
        payload['page_clean_sample_count'] = page_clean.get('sample_count')
        payload['page_clean_page_count'] = page_clean.get('page_count')
        payload['page_clean_mean_cer'] = page_clean.get('mean_cer')
        payload['page_clean_mean_wer'] = page_clean.get('mean_wer')
    payload['run_name'] = run_name
    payload['config_name'] = run_config_name(run_name)
    payload['report_dir'] = str(path.parent)
    summaries.append(payload)
summary_df = pd.DataFrame(summaries)
if summary_df.empty:
    print('No canonical raw OCR evaluation summaries found yet.')
else:
    compact_summary_df = (
        summary_df
        .sort_values(['config_name', 'mean_cer', 'mean_wer'])
        .drop_duplicates('config_name', keep='first')
        .sort_values('mean_cer')
        .reset_index(drop=True)
    )
    display(Markdown('### Best evaluated raw OCR run per config'))
    display(compact_summary_df[['config_name', 'run_name', 'sample_count', 'page_count', 'mean_cer', 'mean_wer', 'page_clean_sample_count', 'page_clean_page_count', 'page_clean_mean_cer', 'page_clean_mean_wer']])
    display(Markdown('### All canonical raw OCR runs'))
    display(summary_df[['run_name', 'config_name', 'sample_count', 'page_count', 'mean_cer', 'mean_wer']].sort_values('mean_cer'))


## Investigate High-Error Predictions


In [ ]:
sample_score_paths = sorted(WORKSPACE.reports_dir.glob('*/per_sample_scores.jsonl'))
if not sample_score_paths:
    print('No per-sample score files found yet.')
else:
    scores = []
    for path in sample_score_paths:
        for row in read_jsonl(path):
            row['report_dir'] = str(path.parent)
            scores.append(row)
    score_df = pd.DataFrame(scores).sort_values('cer', ascending=False)
    display(score_df.head(20)[['report_dir', 'sample_id', 'page_id', 'cer', 'wer', 'reference', 'prediction']])


## Plot High-Error Line Crops


In [ ]:
if 'score_df' not in globals() or score_df.empty or target_df.empty:
    print('No scored predictions to plot yet.')
else:
    row_lookup = target_df.set_index('sample_id').to_dict('index')
    examples = [row for _, row in score_df.head(6).iterrows() if row['sample_id'] in row_lookup]
    fig, axes = plt.subplots(len(examples), 1, figsize=(12, max(2, len(examples) * 1.5)))
    if len(examples) == 1:
        axes = [axes]
    for ax, row in zip(axes, examples):
        image_path = resolve_image_path(row_lookup[row['sample_id']])
        ax.imshow(Image.open(image_path))
        ax.axis('off')
        ax.set_title(f"CER={row['cer']:.3f} | REF: {row['reference'][:60]} | PRED: {row['prediction'][:60]}", fontsize=8)
    plt.tight_layout()


## Plot Correctly Recognized / Low-Error Lines


In [ ]:
if 'score_df' not in globals() or score_df.empty or target_df.empty:
    print('No scored predictions to plot yet.')
else:
    low_df = score_df.sort_values('cer').head(6)
    row_lookup = target_df.set_index('sample_id').to_dict('index')
    fig, axes = plt.subplots(len(low_df), 1, figsize=(12, max(2, len(low_df) * 1.5)))
    if len(low_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, low_df.iterrows()):
        image_path = resolve_image_path(row_lookup[row['sample_id']])
        ax.imshow(Image.open(image_path))
        ax.axis('off')
        ax.set_title(f"CER={row['cer']:.3f} | {row['prediction'][:100]}", fontsize=8)
    plt.tight_layout()


## Final Raw OCR Result Graphs

These graphs keep the raw OCR branch focused on the canonical target-only TrOCR result. The archived An Gaodhal donor branch is excluded by default so the notebook stays aligned with the final target-study path.


In [ ]:
# ## Build plot rows: filter by config, attach page_clean summaries, and keep the canonical target-only raw OCR view.
# Why: plotting the final target-only raw OCR branch makes the held-out OCR baseline easy to compare with correction results downstream.
summary_paths = sorted(WORKSPACE.reports_dir.glob('*/overall_summary.json'))
plot_rows = []
for path in summary_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    run_name = path.parent.name
    if not include_raw_ocr_summary(run_name):
        continue
    page_clean_path = path.parent / 'overall_summary_page_clean.json'
    page_clean = json.loads(page_clean_path.read_text(encoding='utf-8')) if page_clean_path.exists() else {}
    plot_rows.append({
        'run_name': run_name,
        'config_name': run_config_name(run_name),
        'mean_cer': payload.get('mean_cer'),
        'mean_wer': payload.get('mean_wer'),
        'page_clean_mean_cer': page_clean.get('mean_cer'),
        'page_clean_mean_wer': page_clean.get('mean_wer'),
        'sample_count': payload.get('sample_count'),
        'page_count': payload.get('page_count'),
        'page_clean_page_count': page_clean.get('page_count'),
    })

raw_results_df = pd.DataFrame(plot_rows)
if raw_results_df.empty:
    display(Markdown('No canonical raw OCR evaluation summaries found yet. Run the Part 4 evaluation cell first.'))
else:
    best_raw_results_df = (
        raw_results_df
        .sort_values(['config_name', 'mean_cer', 'mean_wer'])
        .drop_duplicates('config_name', keep='first')
        .sort_values('mean_cer')
        .reset_index(drop=True)
    )
    display(Markdown('### Final raw OCR comparison: canonical best run per config'))
    display(best_raw_results_df[['config_name', 'run_name', 'mean_cer', 'mean_wer', 'page_clean_mean_cer', 'page_clean_mean_wer', 'sample_count', 'page_count', 'page_clean_page_count']])
    ensure_dir(WORKSPACE.reports_dir)
    final_csv_path = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_final_raw_ocr_comparison_best_per_config.csv'
    best_raw_results_df.to_csv(final_csv_path, index=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    ordered = best_raw_results_df.sort_values('mean_cer')
    labels = ordered['config_name'].str.replace('trocr_', '', regex=False).str.replace('_pilot', '', regex=False).str.replace('_', ' ', regex=False)
    axes[0].bar(labels, ordered['mean_cer'], label='line-clean test')
    if ordered['page_clean_mean_cer'].notna().any():
        axes[0].scatter(labels, ordered['page_clean_mean_cer'], color='black', marker='D', label='page-clean subset')
    axes[0].set_title('Raw OCR CER by Config')
    axes[0].set_ylabel('Mean CER, lower is better')
    axes[0].tick_params(axis='x', rotation=18)
    axes[0].grid(axis='y', alpha=0.25)
    axes[0].legend()
    ordered_wer = best_raw_results_df.sort_values('mean_wer')
    labels_wer = ordered_wer['config_name'].str.replace('trocr_', '', regex=False).str.replace('_pilot', '', regex=False).str.replace('_', ' ', regex=False)
    axes[1].bar(labels_wer, ordered_wer['mean_wer'], label='line-clean test')
    if ordered_wer['page_clean_mean_wer'].notna().any():
        axes[1].scatter(labels_wer, ordered_wer['page_clean_mean_wer'], color='black', marker='D', label='page-clean subset')
    axes[1].set_title('Raw OCR WER by Config')
    axes[1].set_ylabel('Mean WER, lower is better')
    axes[1].tick_params(axis='x', rotation=18)
    axes[1].grid(axis='y', alpha=0.25)
    axes[1].legend()
    fig.suptitle('HGTCM Canonical Raw OCR Evaluation Results (best run per config)', fontsize=14)
    result_graph_path = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_raw_ocr_result_comparison.png'
    fig.savefig(result_graph_path, dpi=180, bbox_inches='tight')
    plt.show()
    display(Markdown(f'Saved raw OCR result graph: `{result_graph_path}`'))
    display(Markdown(f'Saved compact final comparison CSV: `{final_csv_path}`'))


# Conclusion - Key Takeaways

Use this section after runs complete:

- Report the scratch target TrOCR CER/WER from the canonical held-out OCR branch.
- Report the raw Tesseract Gaelic+Latin baseline from the correction scorecard.
- Report frozen and fine-tuned BART results on the Tesseract branch.
- Report the separate target-only BART control to show whether correction helps or hurts already-strong scratch OCR.
- Keep archived An Gaodhal donor/reference results outside the canonical target-study summary unless you are explicitly discussing the archive experiment.


## Final Correction Scorecard


In [ ]:
# ## latest_correction_summaries: find canonical BART correction report dirs and attach mode-specific files.
CANONICAL_CORRECTION_SOURCES = {
    'target_only': 'scratch_target_trocr',
    'tesseract_gle_lat_target': 'tesseract_gle_lat',
}


def latest_correction_summaries(workspace: Workspace, run_label: str) -> list[dict]:
    paths = sorted(workspace.reports_dir.glob(f'correction_bart_{run_label}_*/overall_summary.json'))
    summaries = []
    mode_summary_files = {
        'frozen_bart_summary': 'corrected_predictions_frozen_bart.summary.json',
        'finetuned_bart_summary': 'corrected_predictions_finetuned_bart.summary.json',
    }
    for path in paths:
        payload = json.loads(path.read_text(encoding='utf-8'))
        source_name = payload.get('correction_source_name')
        if source_name not in CANONICAL_CORRECTION_SOURCES and not INCLUDE_ARCHIVE_REFERENCE_RESULTS:
            continue
        report_dir = path.parent
        payload['report_dir'] = str(report_dir)
        payload['summary_path'] = str(path)
        for key, filename in mode_summary_files.items():
            mode_path = report_dir / filename
            if not mode_path.exists():
                continue
            mode_payload = json.loads(mode_path.read_text(encoding='utf-8'))
            existing = payload.get(key, {}) or {}
            if mode_payload.get('sample_count') and (not existing.get('sample_count') or existing.get('status') == 'not_run'):
                mode_payload['source_mode_summary_path'] = str(mode_path)
                payload[key] = mode_payload
        summaries.append(payload)
    return summaries


def correction_source_label(payload: dict) -> str:
    source_name = payload.get('correction_source_name')
    raw_label = (payload.get('raw_test_summary') or {}).get('label')
    if source_name in CANONICAL_CORRECTION_SOURCES:
        return CANONICAL_CORRECTION_SOURCES[source_name]
    if raw_label == 'raw_scratch_target_trocr':
        return 'scratch_target_trocr'
    if raw_label == 'raw_tesseract_gle_lat':
        return 'tesseract_gle_lat'
    return source_name or raw_label or 'unknown_source'


def rows_from_correction_summary(payload: dict) -> list[dict]:
    rows = []
    raw_summary = payload.get('raw_test_summary', {})
    source_label = correction_source_label(payload)
    source_name = payload.get('correction_source_name') or source_label
    if raw_summary.get('sample_count'):
        rows.append({
            'system': raw_summary.get('label', f'raw_{source_label}'),
            'correction_source_name': source_name,
            'correction_run_id': payload.get('correction_run_id'),
            'sample_count': raw_summary.get('sample_count'),
            'mean_cer': raw_summary.get('mean_cer'),
            'mean_wer': raw_summary.get('mean_wer'),
            'source_summary_path': payload.get('summary_path'),
        })
    for key, system in [
        ('frozen_bart_summary', f'frozen_bart_on_{source_label}'),
        ('finetuned_bart_summary', f'finetuned_bart_on_{source_label}'),
    ]:
        summary = payload.get(key, {})
        if summary.get('sample_count') and 'mean_corrected_cer' in summary:
            rows.append({
                'system': system,
                'correction_source_name': source_name,
                'correction_run_id': payload.get('correction_run_id'),
                'sample_count': summary.get('sample_count'),
                'mean_cer': summary.get('mean_corrected_cer'),
                'mean_wer': summary.get('mean_corrected_wer'),
                'cer_delta_vs_raw': summary.get('cer_delta_raw_minus_corrected'),
                'wer_delta_vs_raw': summary.get('wer_delta_raw_minus_corrected'),
                'source_summary_path': summary.get('source_mode_summary_path') or payload.get('summary_path'),
            })
    return rows


correction_summaries = latest_correction_summaries(WORKSPACE, DATA_RUN_LABEL)
scorecard_rows = []
for payload in correction_summaries:
    scorecard_rows.extend(rows_from_correction_summary(payload))
scorecard_df = pd.DataFrame(scorecard_rows)
if scorecard_df.empty:
    display(Markdown('No canonical BART correction summaries found yet. Run P3 correction first.'))
else:
    best_scorecard_df = (
        scorecard_df
        .sort_values(['system', 'mean_cer', 'mean_wer'])
        .drop_duplicates('system', keep='first')
        .sort_values('mean_cer')
        .reset_index(drop=True)
    )
    scorecard_csv = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_final_correction_scorecard.csv'
    scorecard_json = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_final_correction_scorecard.json'
    best_scorecard_df.to_csv(scorecard_csv, index=False)
    write_json(scorecard_json, best_scorecard_df.to_dict(orient='records'))
    display(Markdown('### Final correction scorecard'))
    display(best_scorecard_df)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    ordered = best_scorecard_df.sort_values('mean_cer')
    axes[0].bar(ordered['system'], ordered['mean_cer'])
    axes[0].set_title('Final CER')
    axes[0].set_ylabel('Mean CER, lower is better')
    axes[0].tick_params(axis='x', rotation=20)
    ordered_wer = best_scorecard_df.sort_values('mean_wer')
    axes[1].bar(ordered_wer['system'], ordered_wer['mean_wer'])
    axes[1].set_title('Final WER')
    axes[1].set_ylabel('Mean WER, lower is better')
    axes[1].tick_params(axis='x', rotation=20)
    plot_path = WORKSPACE.reports_dir / f'{DATA_RUN_LABEL}_final_correction_scorecard.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    display(Markdown(f'Saved final correction scorecard: `{scorecard_csv}` and `{scorecard_json}`.'))
